# Hướng dẫn truy cập và fix cứng môi trường bài toán Portfolios Management và các tinh chỉnh cấu hình

## Khởi tạo môi trường

In [1]:
## install finrl library
!sudo apt install swig
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git




Suggested packages:
  swig-doc swig-examples
The following NEW packages will be installed:
  swig
0 upgraded, 1 newly installed, 0 to remove and 5 not upgraded.
Need to get 1,187 kB of archives.
After this operation, 5,672 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 swig amd64 4.2.0-2ubuntu1 [1,187 kB]
Fetched 1,187 kB in 1s (988 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline

78Selecting previously unselected package swig.
(Reading database ... 125658 files and directories currently installed.)
Preparing to unpack .../swig_4.2.0-2ubuntu1_amd64.deb ...
7Progress: [  0%] [..........................................................] 87Progress: [ 20%] [###########...........................................

In [2]:
## We also need to install quantstats, because the environment uses it to plot graphs
!pip install quantstats

In [3]:
## Hide matplotlib warnings
import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger('matplotlib.font_manager').disabled = True

## Lớp khởi tạo môi trường chính của FinRL

---

Notebook này nhằm mục đích cung cấp một ví dụ về việc sử dụng PortfolioOptimizationEnv (hoặc POE) để huấn luyện một mô hình học tăng cường (reinforcement learning) nhằm giải quyết bài toán tối ưu hóa danh mục đầu tư.

Trong tài liệu này, chúng ta sẽ tái hiện một kiến trúc nổi tiếng có tên là EIIE (Ensemble of Identical Independent Evaluators - tập hợp các bộ đánh giá độc lập giống hệt nhau), được giới thiệu trong bài báo sau:

* Zhengyao Jiang, Dixing Xu, & Jinjun Liang. (2017). A Deep Reinforcement Learning Framework for the Financial Portfolio Management Problem. [https://doi.org/10.48550/arXiv.1706.10059](https://doi.org/10.48550/arXiv.1706.10059).

Bạn nên đọc bài báo này để hiểu rõ thuật toán được triển khai trong notebook.

### Lưu ý

Nếu bạn sử dụng môi trường này, hãy cân nhắc trích dẫn bài báo sau (bên cạnh các tài liệu tham khảo về FinRL):

* Caio Costa, & Anna Costa (2023). POE: A General Portfolio Optimization Environment for FinRL. In *Anais do II Brazilian Workshop on Artificial Intelligence in Finance* (tr. 132–143). SBC. [https://doi.org/10.5753/bwaif.2023.231144](https://doi.org/10.5753/bwaif.2023.231144).

```bibtex
@inproceedings{bwaif,
 author = {Caio Costa and Anna Costa},
 title = {POE: A General Portfolio Optimization Environment for FinRL},
 booktitle = {Anais do II Brazilian Workshop on Artificial Intelligence in Finance},
 location = {João Pessoa/PB},
 year = {2023},
 keywords = {},
 issn = {0000-0000},
 pages = {132--143},
 publisher = {SBC},
 address = {Porto Alegre, RS, Brasil},
 doi = {10.5753/bwaif.2023.231144},
 url = {https://sol.sbc.org.br/index.php/bwaif/article/view/24959}
}

```

In [4]:
!pip install torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.3 MB/s  0:00:00


In [5]:
from typing import Optional, List, Dict, Union, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.preprocessing import MaxAbsScaler

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import GroupByScaler
from finrl.meta.env_portfolio_optimization.env_portfolio_optimization import PortfolioOptimizationEnv
from finrl.agents.portfolio_optimization.models import DRLAgent
from finrl.agents.portfolio_optimization.architectures import EIIE

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

I0000 00:00:1793803949.012245      16 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793803966.553016      16 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Điều chỉnh các đầu vào custom có thể diễn ra

### - Đầu vào chuẩn không áp dụng cho phân tích cảm xúc thị trường (Sentiment Analysis)


In [6]:
# ====================================================================
# ÁP DỤNG TRỰC TIẾP MÔI TRƯỜNG PortfolioOptimizationEnv CHÍNH THỨC CỦA FinRL
# (Costa & Costa 2023, BWAIF - Kế thừa trực tiếp, KHÔNG rewrite lại môi trường)
# ====================================================================

from typing import Optional, List, Dict, Union, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from finrl.meta.env_portfolio_optimization.env_portfolio_optimization import PortfolioOptimizationEnv

class PortfolioOptimizationEnv_v0(PortfolioOptimizationEnv):
    """
    Kế thừa và áp dụng trực tiếp PortfolioOptimizationEnv chính thức của FinRL (Costa & Costa, 2023)
    đã được công bố và kiểm chứng trong file FinRL_PortfolioOptimizationEnv_Demo.ipynb.
    
    Môi trường mô phỏng đầy đủ:
    - Vector biến động giá tương đối y_t = [1.0, p_t / p_{t-1}]
    - Mô hình phí bảo toàn vốn TRF (Transaction Remainder Factor, Jiang et al. 2017)
    - Trôi dạt trọng số thị trường (Price Drift)
    - Phần thưởng log-return: r_t = ln(mu_t * y_t^T * w_t)
    - Không gian quan sát Tensor 3D (F x N x M) và Action Simplex (N + 1)
    """
    def __init__(
        self,
        df: Optional[pd.DataFrame] = None,
        initial_amount: float = 100_000.0,
        comission_fee_pct: float = 0.0025,
        time_window: int = 30,
        features: Optional[List[str]] = None,
        normalize_df: Optional[str] = None,
        return_last_action: bool = False,
        new_gym_api: bool = True,
        # Các tham số hỗ trợ linh hoạt cho cả DataFrame FinRL và ma trận giá
        df_features: Optional[np.ndarray] = None,
        df_prices: Optional[np.ndarray] = None,
        tic_list: Optional[List[str]] = None,
        fee_rate: Optional[float] = None,
        fee_model: str = "trf",
        reward_scaling: float = 1.0,
        **kwargs
    ):
        if features is None:
            features = ["close", "high", "low"]
        if fee_rate is not None:
            comission_fee_pct = fee_rate

        # Chuyển đổi dữ liệu sang định dạng DataFrame chuẩn FinRL nếu truyền ma trận giá
        if df is None and df_prices is not None:
            T_len, N_len = df_prices.shape
            if tic_list is None:
                tic_list = [f"TIC_{i}" for i in range(N_len)]
            date_range = pd.date_range("2020-01-01", periods=T_len, freq="B").strftime("%Y-%m-%d")
            records = []
            for t_idx, d_str in enumerate(date_range):
                for n_idx, tic in enumerate(tic_list):
                    close_p = float(df_prices[t_idx, n_idx])
                    records.append({
                        "date": d_str,
                        "tic": tic,
                        "close": close_p,
                        "high": close_p * 1.002,
                        "low": close_p * 0.998
                    })
            df = pd.DataFrame(records)

        # Gọi trực tiếp khởi tạo của PortfolioOptimizationEnv từ FinRL (KHÔNG rewrite)
        super().__init__(
            df=df,
            initial_amount=initial_amount,
            return_last_action=return_last_action,
            normalize_df=normalize_df,
            reward_scaling=reward_scaling,
            comission_fee_model=fee_model,
            comission_fee_pct=comission_fee_pct,
            features=features,
            time_window=time_window,
            new_gym_api=new_gym_api,
            **kwargs
        )

    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        """Khởi tạo lại môi trường tương thích Gymnasium API mới."""
        if seed is not None:
            self._seed(seed)
        res = super().reset()
        if not self._new_gym_api and not isinstance(res, tuple):
            return res, {}
        return res

    def step(self, actions: np.ndarray):
        """Thực thi một bước mô phỏng với PortfolioOptimizationEnv chính thức."""
        res = super().step(actions)
        if len(res) == 5:
            obs, reward, terminal, truncated, info = res
        else:
            obs, reward, terminal, info = res
            truncated = False
        # Bổ sung các trường tiện ích cho tương thích đánh giá trong notebook
        info["fee_factor"] = info.get("trf_mu", 1.0)
        info["portfolio_value"] = self._portfolio_value
        return obs, reward, terminal, truncated, info

    @property
    def action_dim(self) -> int:
        return self.action_space.shape[0]

    @property
    def num_stocks(self) -> int:
        return self.portfolio_size

    @property
    def time_window(self) -> int:
        return self._time_window

    @property
    def num_features(self) -> int:
        return len(self._features)

    @property
    def T(self) -> int:
        return self.episode_length

    @property
    def asset_memory(self) -> list:
        return self._asset_memory.get("final", [self._initial_amount])

    def compute_metrics(self) -> Dict[str, float]:
        """Tính toán các chỉ số tài chính đánh giá hiệu suất danh mục."""
        values = np.array(self.asset_memory)
        if len(values) < 2:
            return {"Final Value ($)": self._portfolio_value}
        returns = np.diff(values) / values[:-1]
        total_return = (values[-1] - values[0]) / values[0]
        ann_return = ((1.0 + total_return) ** (252.0 / len(values))) - 1.0 if len(values) > 0 else 0.0
        ann_volatility = np.std(returns) * np.sqrt(252.0) if len(returns) > 1 else 0.0
        sharpe = (ann_return - 0.02) / (ann_volatility + 1e-8) if ann_volatility > 0 else 0.0
        cummax = np.maximum.accumulate(values)
        drawdowns = (cummax - values) / cummax
        max_drawdown = np.max(drawdowns) if len(drawdowns) > 0 else 0.0
        return {
            'Initial Value ($)': self._initial_amount,
            'Final Value ($)': values[-1],
            'Cumulative Return (%)': total_return * 100.0,
            'Annualized Return (%)': ann_return * 100.0,
            'Annualized Volatility (%)': ann_volatility * 100.0,
            'Sharpe Ratio': sharpe,
            'Max Drawdown (%)': max_drawdown * 100.0
        }

    def plot_results(self, title: str = "Portfolio Value Over Time"):
        """Hiển thị đồ thị tăng trưởng giá trị danh mục."""
        values = self.asset_memory
        plt.figure(figsize=(10, 5))
        plt.plot(values, label="Portfolio Value", color="#1f77b4", lw=2.0)
        plt.title(title, fontsize=12, fontweight="bold")
        plt.xlabel("Trading Steps", fontsize=10)
        plt.ylabel("Portfolio Value ($)", fontsize=10)
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=10)
        plt.show()

print('✅ Đã áp dụng thành công lớp môi trường PortfolioOptimizationEnv chính thức của FinRL!')

✅ Đã áp dụng thành công lớp môi trường PortfolioOptimizationEnv chính thức của FinRL!


### - Đầu vào cấu hình 1 HISTORICAL PRICE ENVIRONMENT

In [7]:
# ====================================================================
# V1 - HISTORICAL PRICE ENVIRONMENT
#
# Paper:
#   Multimodal Deep Reinforcement Learning for Portfolio Optimization
#
# Configuration:
#   Config 1 - Historical Price Data
#
# Parent:
#   PortfolioOptimizationEnv_v0
#
# Customization:
#   - Override __init__()
#   - Restrict observation features to historical prices
#
# Preserved:
#   - FinRL reset()
#   - FinRL step()
#   - Portfolio accounting
#   - Price drift
#   - TRF transaction cost
#   - Original reward
# ====================================================================

import pandas as pd
from typing import Optional, List


class PortfolioOptimizationEnv_v1(PortfolioOptimizationEnv_v0):

    """
    V1: Historical Price Configuration.

    INPUT:
        df: pandas.DataFrame

        Required columns:
            date
            tic
            close
            high
            low

        Data requirements:
            - Each row represents one asset at one timestamp.
            - All assets must have consistent chronological ordering.
            - Prices must be valid and finite.
            - No sentiment features are included.
            - Data must not contain future information.

    OBSERVATION:
        Historical price tensor generated by FinRL.

        Expected conceptual dimensions:
            (F, N, H)

        F = 3 price features
        N = number of assets
        H = time_window

        Actual axis order must be verified against FinRL runtime.

    ACTION:
        Same action space as V0.

    OUTPUT:
        reset():
            observation, info

        step():
            observation,
            reward,
            terminated,
            truncated,
            info

        Portfolio dynamics and reward remain inherited from V0.
    """

    # ---------------------------------------------------------------
    # V1 INPUT CONFIGURATION
    # ---------------------------------------------------------------

    V1_FEATURES = [
        "close",
        "high",
        "low",
    ]

    REQUIRED_COLUMNS = [
        "date",
        "tic",
        "close",
        "high",
        "low",
    ]

    # ---------------------------------------------------------------
    # OVERRIDE __init__
    # ---------------------------------------------------------------

    def __init__(
        self,
        df: pd.DataFrame,

        # FinRL original settings
        initial_amount: float = 100_000.0,
        comission_fee_pct: float = 0.0025,
        time_window: int = 30,

        normalize_df: Optional[str] = None,
        return_last_action: bool = False,
        new_gym_api: bool = True,

        **kwargs
    ):

        # ===========================================================
        # INPUT VALIDATION
        # ===========================================================

        if df is None:
            raise ValueError("V1 requires a DataFrame.")

        missing = [
            col for col in self.REQUIRED_COLUMNS
            if col not in df.columns
        ]

        if missing:
            raise ValueError(
                f"V1 missing columns: {missing}"
            )

        # ===========================================================
        # V1 CUSTOM FEATURES
        # ===========================================================

        features = self.V1_FEATURES.copy()

        # ===========================================================
        # INHERIT FINRL ENVIRONMENT
        # ===========================================================

        super().__init__(
            df=df,
            initial_amount=initial_amount,
            comission_fee_pct=comission_fee_pct,
            time_window=time_window,
            features=features,
            normalize_df=normalize_df,
            return_last_action=return_last_action,
            new_gym_api=new_gym_api,
            **kwargs
        )

        # -----------------------------------------------------------
        # V1 METADATA
        # -----------------------------------------------------------

        self.version = "v1"
        self.configuration = "historical_price"
        self.custom_features = features

### - Đầu vào cấu hình 2 HISTORICAL PRICE + SEC SENTIMENT ENVIRONMENT

In [8]:
# ====================================================================
# V2 - PRICE + SEC SENTIMENT ENVIRONMENT
#
# Paper:
#   Multimodal Deep Reinforcement Learning for Portfolio Optimization
#
# Configuration:
#   Config 2 - Historical Price + SEC Sentiment
#
# Parent:
#   PortfolioOptimizationEnv_v0
#
# Customization:
#   - Override __init__()
#   - Add SEC sentiment feature
#
# Preserved:
#   - FinRL portfolio dynamics
#   - TRF transaction cost
#   - Price drift
#   - FinRL reward
#   - FinRL action processing
# ====================================================================


import pandas as pd
from typing import Optional, List


class PortfolioOptimizationEnv_v2(PortfolioOptimizationEnv_v0):

    """
    V2: Price + SEC Sentiment.

    INPUT:
        pandas.DataFrame

        Required columns:
            date
            tic
            close
            high
            low
            sec_sentiment

    SEC INPUT RULES:
        - Sentiment must be computed before environment initialization.
        - SEC scores must be aligned by ticker and date.
        - Sentiment must be available at decision time.
        - No future filing information is allowed.
        - Missing-date values should follow the paper's
          exponential decay procedure.

    OBSERVATION:
        Historical price features + SEC sentiment.

        Conceptual dimensions:
            (F, N, H)

        F = 4 features
        N = number of assets
        H = time_window

    ACTION:
        Same as V0.

    REWARD:
        Same as V0.

    OUTPUT:
        Same Gymnasium interface as V0.
    """

    # ---------------------------------------------------------------
    # V2 CUSTOM FEATURES
    # ---------------------------------------------------------------

    V2_FEATURES = [
        "close",
        "high",
        "low",
        "sec_sentiment",
    ]

    REQUIRED_COLUMNS = [
        "date",
        "tic",
        "close",
        "high",
        "low",
        "sec_sentiment",
    ]

    # ---------------------------------------------------------------
    # OVERRIDE __init__
    # ---------------------------------------------------------------

    def __init__(
        self,
        df: pd.DataFrame,

        # FinRL original settings
        initial_amount: float = 100_000.0,
        comission_fee_pct: float = 0.0025,
        time_window: int = 30,

        normalize_df: Optional[str] = None,
        return_last_action: bool = False,
        new_gym_api: bool = True,

        **kwargs
    ):

        # ===========================================================
        # INPUT VALIDATION
        # ===========================================================

        if df is None:
            raise ValueError("V2 requires a DataFrame.")

        missing = [
            col for col in self.REQUIRED_COLUMNS
            if col not in df.columns
        ]

        if missing:
            raise ValueError(
                f"V2 missing columns: {missing}"
            )

        # ===========================================================
        # SEC SENTIMENT VALIDATION
        # ===========================================================

        if df["sec_sentiment"].isna().any():
            raise ValueError(
                "V2 requires SEC sentiment values to be "
                "preprocessed and aligned."
            )

        # ===========================================================
        # V2 CUSTOM FEATURES
        # ===========================================================

        features = self.V2_FEATURES.copy()

        # ===========================================================
        # INHERIT FINRL ENVIRONMENT
        # ===========================================================

        super().__init__(
            df=df,
            initial_amount=initial_amount,
            comission_fee_pct=comission_fee_pct,
            time_window=time_window,
            features=features,
            normalize_df=normalize_df,
            return_last_action=return_last_action,
            new_gym_api=new_gym_api,
            **kwargs
        )

        # -----------------------------------------------------------
        # V2 METADATA
        # -----------------------------------------------------------

        self.version = "v2"
        self.configuration = "price_sec"
        self.custom_features = features

### - Đầu vào cấu hình 3 HISTORICAL PRICE + SEC + NEWS SENTIMENT ENVIRONMENT

In [9]:
# ====================================================================
# V3 - PRICE + SEC + NEWS SENTIMENT ENVIRONMENT
#
# Paper:
#   Multimodal Deep Reinforcement Learning for Portfolio Optimization
#
# Configuration:
#   Config 3 - Historical Price + SEC + News Sentiment
#
# Parent:
#   PortfolioOptimizationEnv_v0
#
# Customization:
#   - Override __init__()
#   - Add SEC sentiment
#   - Add News sentiment
#
# Preserved:
#   - FinRL portfolio accounting
#   - TRF transaction cost
#   - Price drift
#   - Original reward
#   - Original action processing
# ====================================================================


import pandas as pd
from typing import Optional, List


class PortfolioOptimizationEnv_v3(PortfolioOptimizationEnv_v0):

    """
    V3: Price + SEC + News Sentiment.

    INPUT:
        pandas.DataFrame

        Required columns:
            date
            tic
            close
            high
            low
            sec_sentiment
            news_sentiment

    SEC INPUT:
        - SEC sentiment must be computed in preprocessing.
        - SEC score must be aligned by ticker/date.
        - Exponential decay must be applied according to
          the paper's preprocessing procedure.

    NEWS INPUT:
        - News sentiment must be generated before environment
          initialization.
        - FinBERT probabilities are processed into sentiment score.
        - News sentiment must be aligned by ticker/date.
        - Exponential decay handles missing news dates.
        - Future news must not enter current observations.

    OBSERVATION:
        Historical prices + SEC + News.

        Conceptual dimensions:
            (F, N, H)

        F = 5 features
        N = number of assets
        H = time_window

    ACTION:
        Same as V0.

    REWARD:
        Same as V0.

    OUTPUT:
        Same Gymnasium interface as V0.
    """

    # ---------------------------------------------------------------
    # V3 CUSTOM FEATURES
    # ---------------------------------------------------------------

    V3_FEATURES = [
        "close",
        "high",
        "low",
        "sec_sentiment",
        "news_sentiment",
    ]

    REQUIRED_COLUMNS = [
        "date",
        "tic",
        "close",
        "high",
        "low",
        "sec_sentiment",
        "news_sentiment",
    ]

    # ---------------------------------------------------------------
    # OVERRIDE __init__
    # ---------------------------------------------------------------

    def __init__(
        self,
        df: pd.DataFrame,

        # FinRL original settings
        initial_amount: float = 100_000.0,
        comission_fee_pct: float = 0.01,
        time_window: int = 30,

        normalize_df: Optional[str] = None,
        return_last_action: bool = False,
        new_gym_api: bool = True,

        **kwargs
    ):

        # ===========================================================
        # INPUT VALIDATION
        # ===========================================================

        if df is None:
            raise ValueError("V3 requires a DataFrame.")

        missing = [
            col for col in self.REQUIRED_COLUMNS
            if col not in df.columns
        ]

        if missing:
            raise ValueError(
                f"V3 missing columns: {missing}"
            )

        # ===========================================================
        # SENTIMENT VALIDATION
        #
        # Data preprocessing must define how missing sentiment
        # values are handled.
        #
        # Do not silently use future sentiment.
        # ===========================================================

        sentiment_columns = [
            "sec_sentiment",
            "news_sentiment",
        ]

        for col in sentiment_columns:

            if df[col].isna().any():

                raise ValueError(
                    f"V3 requires preprocessing of missing "
                    f"sentiment values in '{col}'."
                )

        # ===========================================================
        # V3 CUSTOM FEATURES
        # ===========================================================

        features = self.V3_FEATURES.copy()

        # ===========================================================
        # INHERIT FINRL ENVIRONMENT
        # ===========================================================

        super().__init__(
            df=df,
            initial_amount=initial_amount,
            comission_fee_pct=comission_fee_pct,
            time_window=time_window,
            features=features,
            normalize_df=normalize_df,
            return_last_action=return_last_action,
            new_gym_api=new_gym_api,
            **kwargs
        )

        # -----------------------------------------------------------
        # V3 METADATA
        # -----------------------------------------------------------

        self.version = "v3"
        self.configuration = "price_sec_news"
        self.custom_features = features

### - Môi trường đặc trưng cho FinRL kết hợp 2 thành phần phần thưởng mới

In [10]:
import numpy as np
import pandas as pd

from finrl.meta.env_portfolio_optimization.env_portfolio_optimization import (
    PortfolioOptimizationEnv
)


class PortfolioZeroRewardPOE(PortfolioOptimizationEnv):
    """
    FinRL PortfolioOptimizationEnv + PortfolioZero-style sentiment rewards.

    Kế thừa:
        - State, reset, action normalization
        - Transaction fee
        - Portfolio value accounting
        - Price variation / portfolio weights

    Override:
        - Reward R1, R2, R3
    """

    def __init__(
        self,
        *args,
        sentiment_df=None,
        reward_mode="R1",
        sentiment_weight=1.0,
        beta_weight=1.0,
        **kwargs
    ):
        super().__init__(*args, **kwargs)

        if reward_mode not in {"R1", "R2", "R3"}:
            raise ValueError("reward_mode phải là R1, R2 hoặc R3")

        self.reward_mode = reward_mode
        self.sentiment_weight = sentiment_weight
        self.beta_weight = beta_weight

        # Data format: date, tic, score
        if sentiment_df is None:
            sentiment_df = pd.DataFrame(
                columns=["date", "tic", "score"]
            )

        sentiment_df = sentiment_df.copy()
        sentiment_df["date"] = pd.to_datetime(
            sentiment_df["date"]
        )

        self.sentiment_df = sentiment_df

        # Lookup: (date, ticker) -> sentiment score
        self.sentiment_lookup = {
            (row.date, row.tic): float(row.score)
            for row in sentiment_df.itertuples(index=False)
        }

        # Logging reward components
        self.reward_components = []

    def _get_sentiment_vector(self, date):
        """
        Trả về sentiment theo đúng thứ tự self._tics.

        POE action convention:
            index 0 = cash
            index 1..N = risky assets

        Cash sentiment được gán bằng 0.
        """
        date = pd.to_datetime(date)

        sentiment = np.zeros(len(self._tics) + 1, dtype=np.float32)

        for i, tic in enumerate(self._tics, start=1):
            sentiment[i] = self.sentiment_lookup.get(
                (date, tic), 0.0
            )

        return sentiment

    @staticmethod
    def _rank_weights_change(delta_weights):
        """
        Rank theo mức thay đổi weights.

        Rank nhỏ: mức thay đổi thấp hơn.
        Rank lớn: mức thay đổi cao hơn.

        Ties được xử lý bằng average rank.
        """
        return (
            pd.Series(delta_weights)
            .rank(method="average")
            .to_numpy(dtype=np.float32)
        )

    def _calculate_sentiment_rewards(
        self,
        weights,
        previous_weights,
        sentiment
    ):
        """
        weights:
            Target weights agent vừa yêu cầu.

        previous_weights:
            Weights trước khi thực hiện action.

        sentiment:
            Sentiment vector gồm cash + stocks.
        """

        # Exclude cash index 0
        stock_weights = weights[1:]
        stock_previous_weights = previous_weights[1:]
        stock_sentiment = sentiment[1:]

        # R_alpha: sentiment-weighted allocation
        R_alpha = np.sum(
            stock_weights * stock_sentiment
        )

        # Delta portfolio weights
        delta_weights = (
            stock_weights - stock_previous_weights
        )

        # Rank delta weights
        ranks = self._rank_weights_change(delta_weights)

        # R_beta: pairwise sentiment differences * rank differences
        sentiment_diff = (
            stock_sentiment[:, None]
            - stock_sentiment[None, :]
        )

        rank_diff = (
            ranks[:, None]
            - ranks[None, :]
        )

        R_beta = np.sum(
            sentiment_diff * rank_diff
        )

        return float(R_alpha), float(R_beta)

    def step(self, actions):
        """
        POE step + custom PortfolioZero reward.

        Base POE vẫn xử lý:
            action normalization
            transaction fees
            portfolio value
            price variation
            next state
            terminal condition
        """

        # Base environment step
        result = super().step(actions)

        # Gym API compatibility
        if self._new_gym_api:
            state, base_reward, terminal, truncated, info = result
        else:
            state, base_reward, terminal, info = result

        # Nếu POE đã terminal thì không có action trading mới
        if terminal:
            return result

        # Base reward trước reward_scaling
        # PortfolioZero R = log(V_t / V_{t-1})
        base_log_return = np.log(
            self._asset_memory["final"][-1]
            / self._asset_memory["final"][-2]
        )

        # Current target action weights
        current_weights = np.asarray(
            self._actions_memory[-1],
            dtype=np.float32
        )

        # Portfolio weights before action
        previous_weights = np.asarray(
            self._final_weights[-2],
            dtype=np.float32
        )

        # Sentiment date corresponding to this transition
        date = pd.to_datetime(info["end_time"])

        sentiment = self._get_sentiment_vector(date)

        R_alpha, R_beta = self._calculate_sentiment_rewards(
            weights=current_weights,
            previous_weights=previous_weights,
            sentiment=sentiment
        )

        # Reward composition
        if self.reward_mode == "R1":
            reward_unscaled = base_log_return

        elif self.reward_mode == "R2":
            reward_unscaled = (
                base_log_return
                + self.sentiment_weight * R_alpha
            )

        elif self.reward_mode == "R3":
            reward_unscaled = (
                base_log_return
                + self.sentiment_weight * R_alpha
                + self.beta_weight * R_beta
            )

        # Apply POE reward scaling exactly once
        reward = reward_unscaled * self._reward_scaling

        # Override reward from base POE
        self._reward = reward

        # Update reward memory so POE's episode metrics use custom reward
        self._portfolio_reward_memory[-1] = reward_unscaled

        # Logging
        reward_log = {
            "date": date,
            "base_log_return": float(base_log_return),
            "R_alpha": float(R_alpha),
            "R_beta": float(R_beta),
            "reward_unscaled": float(reward_unscaled),
            "reward_scaled": float(reward),
            "reward_mode": self.reward_mode
        }

        self.reward_components.append(reward_log)

        # Add components to info
        info = dict(info)
        info.update(reward_log)

        if self._new_gym_api:
            return state, reward, terminal, truncated, info

        return state, reward, terminal, info

## Truyền mô hình và scoring sentiment

In [11]:
import torch
import math
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Khởi tạo mô hình và tokenizer FinBERT
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [12]:
def calculate_custom_sentiment_score(text):
    # 2. Tiền xử lý văn bản
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    
    # 3. Đưa qua mô hình để lấy raw logits
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        
    # 4. Áp dụng hàm Softmax để tính xác suất (probabilities)
    probabilities = torch.nn.functional.softmax(logits, dim=-1).squeeze().tolist()
    
    # Thứ tự nhãn của ProsusAI/finbert là: [positive, negative, neutral]
    # Lưu ý: Bạn cần kiểm tra lại model.config.id2label để chắc chắn thứ tự nhãn
    positive_prob = probabilities[0]
    negative_prob = probabilities[1]
    neutral_prob = probabilities[2]
    
    # Xử lý trường hợp negative_prob hoặc neutral_prob quá nhỏ (tránh chia cho 0)
    epsilon = 1e-9
    negative_prob = max(negative_prob, epsilon)
    neutral_prob = max(neutral_prob, epsilon)
    
    # 5. Áp dụng công thức (1) từ bài báo
    ratio = (positive_prob / negative_prob) / neutral_prob
    value_embedding = math.tanh(ratio)
    
    return {
        "text": text,
        "positive_prob": positive_prob,
        "negative_prob": negative_prob,
        "neutral_prob": neutral_prob,
        "scoring_value": value_embedding
    }

In [13]:
# --- Chạy thử trên một tập mẫu (Sample) ---
sample_headlines = [
    "Apple iPhone sales drop at recent pace!",
    "Powerful 11th Generation Intel Core CPUs Are Coming.",
    "The company's quarterly revenue met analysts' expectations."
]

for headline in sample_headlines:
    result = calculate_custom_sentiment_score(headline)
    print(f"Headline: '{result['text']}'")
    print(f"  - Positive: {result['positive_prob']:.4f}")
    print(f"  - Negative: {result['negative_prob']:.4f}")
    print(f"  - Neutral:  {result['neutral_prob']:.4f}")
    print(f"  -> Scoring Value (value_embedding): {result['scoring_value']:.4f}\\n")

Headline: 'Apple iPhone sales drop at recent pace!'
  - Positive: 0.0076
  - Negative: 0.9709
  - Neutral:  0.0215
  -> Scoring Value (value_embedding): 0.3510\n
Headline: 'Powerful 11th Generation Intel Core CPUs Are Coming.'
  - Positive: 0.5014
  - Negative: 0.0132
  - Neutral:  0.4854
  -> Scoring Value (value_embedding): 1.0000\n
Headline: 'The company's quarterly revenue met analysts' expectations.'
  - Positive: 0.9544
  - Negative: 0.0179
  - Neutral:  0.0277
  -> Scoring Value (value_embedding): 1.0000\n
